In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pyspark.sql import functions as F

catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = spark.sql("SELECT current_schema()").first()[0]

RAW_TABLE = f"{catalog_name}.{schema_name}.fraud_transactions_raw"

df = spark.table(RAW_TABLE)

print("Source table:", RAW_TABLE)
print("Rows:", df.count())
print("Columns:", len(df.columns))

display(df.limit(5))

Source table: workspace.default.fraud_transactions_raw
Rows: 284807
Columns: 31


Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
127157.0,0.095972576,1.104455024,-0.194498984,-0.437905402,0.786594139,-1.073488179,1.075507285,-0.207669323,-0.257236309,-0.873206264,-0.081283261,0.65428777,1.095596456,-1.22094371,-0.400082171,0.143927807,0.466018063,-0.506221443,-0.410941782,0.095151795,-0.29353789,-0.655851824,0.154528016,1.074258109,-0.439686082,0.079642611,0.228074611,0.092228208,8.99,0
127157.0,-0.335184307,0.482852176,0.742958697,-0.577091328,0.60780573,-0.460894852,1.110537073,-0.737622366,0.75803995,0.052796774,-0.238322856,0.782401321,1.26261705,-0.862135029,0.014230587,-0.452585107,-0.725226256,-0.464183811,-0.452738889,-0.239312738,0.025280719,0.652235615,0.133099729,1.122989367,-0.306402037,-0.559992363,-0.695872365,-0.496800362,26.63,0
127157.0,1.954634458,-0.583886948,-1.454837504,0.162864701,0.261320198,0.447738138,-0.376322971,0.206034214,0.868919434,0.281433081,-0.428359407,-0.419458654,-1.835577994,0.657242862,-0.046475963,0.660729587,-0.915415135,0.656884867,0.498390039,-0.16728918,-0.044821636,-0.253612998,0.093887709,-0.442842212,-0.171206183,0.246080128,-0.070027554,-0.062692577,60.9,0
127157.0,-1.425761815,-0.642619026,0.790258428,-2.353993556,-0.200552762,0.063102237,-0.600192206,0.763625362,-1.030356484,-0.188432302,0.15727345,-0.417773364,-0.134998215,-0.039587817,-0.66366129,1.986052232,-0.460277956,-0.155105305,-0.018629788,0.442471199,0.574900196,1.10949687,-0.099477441,0.279512764,0.172515968,-0.152024915,0.200716274,0.015917862,95.0,0
127158.0,-0.314367339,1.112116695,0.478770903,0.068965784,1.169654094,-0.442801922,1.206612347,-0.167853942,-0.747024525,-0.511983397,-0.987118852,-0.072292823,0.326854646,0.332108695,0.168137775,-0.115758131,-0.701175923,0.083705991,-0.724730356,-0.113576395,0.347086869,1.022908186,-0.367024808,0.693068444,0.232445346,-0.489739601,0.148036998,0.161970273,6.17,0


In [0]:
assert "Class" in df.columns, "Target column 'Class' is missing"

display(df.groupBy("Class").count().orderBy("Class"))

target_nulls = df.filter(F.col("Class").isNull()).count()
assert target_nulls == 0, f"Target column 'Class' has {target_nulls} null values"

classes = {
    row["Class"]
    for row in df.select("Class").distinct().collect()
}

assert classes == {0,1}, f"Unexpected target classes: {classes}"

print("Target validation passed")

Class,count
0,284315
1,492


Target validation passed


In [0]:
pdf = df.toPandas()
X = pdf.drop(columns = ["Class"])
y = pdf["Class"].astype("int32")

assert X.select_dtypes(exclude="number").empty, "Non-numeric features found; inspect the schema before proceeding"

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training rows", len(X_train))
print("Testing rows", len(X_test))
print("Training fraud rate", round(y_train.mean(),6))

Training rows 227845
Testing rows 56962
Training fraud rate 0.001729


In [0]:
TRAIN_TABLE = f"{catalog_name}.{schema_name}.fraud_transaction_train"
TEST_TABLE = f"{catalog_name}.{schema_name}.fraud_transaction_test"

train_pdf = X_train.copy()
train_pdf["Class"] = y_train

test_pdf = X_test.copy()
test_pdf["Class"] = y_test

spark.createDataFrame(train_pdf).write.format("delta").mode("overwrite").saveAsTable(TRAIN_TABLE)
spark.createDataFrame(test_pdf).write.format("delta").mode("overwrite").saveAsTable(TEST_TABLE)

print("Training table:", TRAIN_TABLE)
print("Testing table:", TEST_TABLE)


Training table: workspace.default.fraud_transaction_train
Testing table: workspace.default.fraud_transaction_test


In [0]:
train_df = spark.table(TRAIN_TABLE)
test_df = spark.table(TEST_TABLE)

print("Training rows", train_df.count())
print("Testing rows", test_df.count())

print("Training class distribution")
display(train_df.groupBy("Class").count().orderBy("Class"))

print("Testing class distribution")
display(test_df.groupBy("Class").count().orderBy("Class"))
assert train_df.count() + test_df.count() == len(pdf), "The split row count do not match the source dataset"


Training rows 227845
Testing rows 56962
Training class distribution


Class,count
0,227451
1,394


Testing class distribution


Class,count
0,56864
1,98
